# Apple India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** jobs.apple.com/en-in/search?location=india-INDC

**ATS Detection:** Automatic API + Selenium fallback

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml playwright -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path

# Add scripts dir to path so we can import scraper_utils
SCRIPTS_DIR = Path.home() / "Job_Scrapers" / "All_Scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

from scraper_utils import *
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

# ── LOCATION CONFIG ──────────────────────────────────────────────────────────
# Change to "" to scrape globally (all countries).
# The matching pipeline's pre-filter handles India-specific narrowing.
# Set to "India" here only if you want to reduce volume at scrape time.
LOCATION_FILTER = ""
COUNTRY_CODE   = ""  # e.g. "in" for SmartRecruiters country= param; "" = all
# ─────────────────────────────────────────────────────────────────────────────

print("Imports loaded. Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Location filter: '{LOCATION_FILTER}' (empty = broad/global scraping)")


scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills
Imports loaded. Date: 2026-03-31 21:27:39
Location filter: '' (empty = broad/global scraping)


In [3]:
COMPANY = "Apple"
OUTPUT_DIR = get_output_dir(COMPANY)
print(f"Output directory: {OUTPUT_DIR}")


Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/Apple/Outputs/2026_03_31


In [4]:
print("=" * 60)
print("APPLE INDIA JOB SCRAPER")
print("Source: jobs.apple.com/en-in/search?location=india-INDC")
print("DOM: div.job-list-item > a.link-inline[href*='/details/']")
print("=" * 60)

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


def fetch_jd_selenium(driver, url, timeout=10):
    """Visit a job detail page and extract the JD text."""
    try:
        driver.get(url)
        time.sleep(random.uniform(2, 4))
        soup = BeautifulSoup(driver.page_source, "lxml")
        # Try common JD container selectors
        for sel in ["[class*='job-description']", "[class*='jd-info']", "[class*='description']",
                     "[class*='details']", "article", "main", ".content"]:
            el = soup.select_one(sel)
            if el and len(el.get_text(strip=True)) > 100:
                return el.get_text(" ", strip=True)
        # Fallback: get body text
        body = soup.select_one("body")
        return body.get_text(" ", strip=True)[:5000] if body else ""
    except Exception as e:
        print(f"    [WARN] JD fetch failed for {url}: {e}")
        return ""

def fetch_jd_requests(session, url):
    """Fetch a job detail page via requests and extract JD text."""
    try:
        resp = session.get(url, timeout=20)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "lxml")
            for sel in ["[class*='job-description']", "[class*='jd-info']", "[class*='description']",
                         "[class*='details']", "article", "main"]:
                el = soup.select_one(sel)
                if el and len(el.get_text(strip=True)) > 100:
                    return el.get_text(" ", strip=True)
            body = soup.select_one("body")
            return body.get_text(" ", strip=True)[:5000] if body else ""
    except:
        pass
    return ""


apple_jobs = []
driver = setup_selenium()

try:
    base_url = "https://jobs.apple.com/en-in/search?location=india-INDC"
    driver.get(base_url)
    time.sleep(8)

    # Wait for job cards to render
    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.job-list-item"))
        )
    except:
        print("  Waiting longer for Apple jobs page to load...")
        time.sleep(10)

    page = 1
    while page <= 15:
        soup = BeautifulSoup(driver.page_source, "lxml")

        # Apple uses div.job-list-item for each job card (confirmed Mar 2026)
        cards = soup.select("div.job-list-item")
        if not cards and page == 1:
            # Try alternate selectors
            cards = soup.select("a[href*='/en-in/details/']")
            cards = [c.parent for c in cards if c.parent]

        if not cards:
            print(f"  Page {page}: No job cards found")
            if page == 1:
                print(f"  Page title: {driver.title}")
            break

        new_jobs = 0
        for card in cards:
            # Title: a.link-inline with href to /details/
            title_link = card.select_one("a.link-inline[href*='/details/'], a[href*='/details/']")
            if not title_link:
                continue
            title = title_link.get_text(strip=True)
            href = title_link.get("href", "")

            # Extract job ID from URL: /en-in/details/200314122/job-slug
            job_id_match = re.search(r"/details/(\d+)/", href)
            job_id = job_id_match.group(1) if job_id_match else href.split("/")[-1]

            # Team name: span.team-name
            team_el = card.select_one("span.team-name")
            team = team_el.get_text(strip=True) if team_el else ""

            # Posted date: span.job-posted-date
            date_el = card.select_one("span.job-posted-date")
            date_text = date_el.get_text(strip=True) if date_el else ""
            # Parse date like "21 Mar 2026" -> "2026-03-21"
            posted_date = datetime.now().strftime("%Y-%m-%d")
            if date_text:
                try:
                    posted_date = datetime.strptime(date_text, "%d %b %Y").strftime("%Y-%m-%d")
                except:
                    pass

            # Location: look for text after "Location"
            loc_spans = card.select("span")
            loc = "India"
            for span in loc_spans:
                text = span.get_text(strip=True)
                if any(city in text for city in ["Bengaluru", "Mumbai", "Hyderabad", "Pune",
                       "Chennai", "Delhi", "Gurugram", "Noida", "Kolkata", "India"]):
                    loc = text
                    break

            if title and title not in [j["title"] for j in apple_jobs]:
                apple_jobs.append({
                    "job_id": job_id,
                    "title": title,
                    "company_name": "Apple",
                    "raw_jd_text": "",  # Will fetch individually below
                    "location_city": loc.split(",")[0].strip().replace("Various locations within ", ""),
                    "industry": "Technology",
                    "date_posted": posted_date,
                    "is_active": True,
                    "job_url": f"https://jobs.apple.com{href}",
                    "business_unit": team,
                    "source_platform": "Apple Jobs",
                })
                new_jobs += 1

        print(f"  Page {page}: {new_jobs} new jobs (total: {len(apple_jobs)})")

        if new_jobs == 0:
            break

        # Click Next page button
        try:
            next_btn = driver.find_element(By.CSS_SELECTOR,
                "button[aria-label='Next Page']:not([disabled])")
            driver.execute_script("arguments[0].click();", next_btn)
            time.sleep(4)
            page += 1
        except:
            break

    # Fetch JD for first N jobs (limit to avoid being blocked)
    print(f"\n  Fetching JD details for up to 50 jobs...")
    for i, job in enumerate(apple_jobs[:50]):
        if job["raw_jd_text"]:
            continue
        detail_url = f"https://jobs.apple.com/en-in/details/{job['job_id']}"
        jd = fetch_jd_selenium(driver, detail_url)
        apple_jobs[i]["raw_jd_text"] = jd
        if (i + 1) % 10 == 0:
            print(f"    Fetched {i+1}/{min(50, len(apple_jobs))} JDs")

except Exception as e:
    print(f"  Error: {e}")
    import traceback; traceback.print_exc()
finally:
    driver.quit()

print(f"Total Apple India jobs: {len(apple_jobs)}")
has_jd = sum(1 for j in apple_jobs if j.get("raw_jd_text") and len(j["raw_jd_text"]) > 50)
print(f"  Jobs with JD: {has_jd}/{len(apple_jobs)}")


APPLE INDIA JOB SCRAPER
Source: jobs.apple.com/en-in/search?location=india-INDC
DOM: div.job-list-item > a.link-inline[href*='/details/']


  Page 1: 19 new jobs (total: 19)


  Page 2: 19 new jobs (total: 38)


  Page 3: 20 new jobs (total: 58)


  Page 4: 15 new jobs (total: 73)


...

    Fetched 10/50 JDs


    Fetched 20/50 JDs


    Fetched 30/50 JDs


    Fetched 40/50 JDs


    Fetched 50/50 JDs
Total Apple India jobs: 208
  Jobs with JD: 50/208


In [5]:
df_apple = save_results(apple_jobs, "Apple", OUTPUT_DIR)
if df_apple is not None:
    print(f"\nSample jobs:")
    cols = ["title","location_city","seniority_level","business_unit","job_url"]
    cols = [c for c in cols if c in df_apple.columns]
    print(df_apple[cols].head(10).to_string())


  [OK] Saved 208 jobs -> Apple_jobs_2026-03-31.csv
       Seniority: {'mid': 112, 'lead': 79, 'senior': 12, 'junior': 5}
       Work mode: {'onsite': 167, 'remote': 41}
       Has JD text: 50/208
       Has job URL: 208/208
       Has business unit: 207/208

Sample jobs:
                                                            title location_city seniority_level                business_unit                                                                                                              job_url
0  IN - Specialist: Full-Time, Part-Time, and Part-Time Temporary         India          senior                 Apple Retail  https://jobs.apple.com/en-in/details/200314117/in-specialist-full-time-part-time-and-part-time-temporary?team=APPST
1                                       See full role description         India          senior                               https://jobs.apple.com/en-in/details/200314117/in-specialist-full-time-part-time-and-part-time-temporary?team=APPST
2 